In [2]:
import pandas as pd
import numpy as np
import datetime as dt

# 날짜와 시간의 연산



모든 데이터 분석이 그렇지만 데이터 분석을 위해서는 분석 대상 데이터를 가져온 이후, 데이터의 모델링과 분석, 예측에 적합하도록 데이터를 전처리 해야 한다. 일반적 수치 데이터(numerical data), 문자 데이터(character data), 범주형 데이터(categorical data) 등과 달리 시계열 데이터는 달력 상의 날짜에 대한 연산 처리, 24시간, 60분과 같은 일정치 않은 주기에 대한 연산이 필요하다. python에서 날짜와 시간에 대한 연산이 어떻게 수행되는지를 이해하기 위해서는 날짜와 시간의 연산에 사용되는 데이터 클래스인 dateoffset, timedelta, peroid를 이해해야 한다. 

## dateoffset을 사용한 연산

dateoffset은 pandas에서 제공하는 날짜와 시간의 간격을 표현하는데 전용으로 사용되는 데이터 클래스이다. dateoffset은 달력상의 날짜와 시간을 기반으로 두 날짜나 시간 간의 차이를 표현하는데 특정 두 시점의 시작부터 끝까지의 차이가 아닌 일반적인 날짜와 시간의 간격을 의미한다. DateOffset은 "days", "weeks", "years"와 같은 키워드를 사용하여 날짜를 일, 주, 년씩 증가시키거나 감소시킬수 있다. dateoffset을 생성하기 위해서는 pandas의 `dateoffset()`을 사용하여 생성할 수 있는데, 다음은 `dataoffset()`에서 날짜 간격을 설정하는데 사용하는 주요 매개변수이다. 
|키워드|주기|
|---|---|
|years|연도|
|months|월|
|weeks|주|
|days|일|
|hours|시간|
|minutes|분|
|seconds|초|
|milliseconds|밀리초|
|microseconds|마이크로초|
|nanoseconds|나노초|

dateoffset에서는 유용한 서브 클래스를 제공한다. 이중 매우 유용하게 사용되는 서브 클래스는 달력상의 날짜가 아닌 영업일 상의 날짜와 시간을 계산하는 business 관련 dateoffset이다. 주요 business 관련 dateoffset의 서브 클래스는 다음과 같다.
|서브 클래스 명|설명|별칭|주요 속성|주요 메쏘드|
|---|---|---|---|---|
|BusinessDay|영업일을 표현하는데 사용되는 서브 클래스|BDay|freqstr, name, holidays|copy(), is_on_offset(), is_month_start(), is_month_end()|
|BusinessHour|영업시간을 표현하는데 사용되는 서브 클래스||freqstr, name, next_bday|copy(), is_on_offset(), is_month_start(), is_month_end(), rollback(), rollforward()|
|FY5253|52-53주의 회계 연도를 표현하는데 사용되는 서브 클래스||freqstr, name|copy(), is_month_start(), is_month_end(), rollback(), rollforward()|

다음은 dateoffset의 서브 클래스인 BusinessDay를 사용하는 예를 보이고 있다. 2023년 2월 5일은 일요일이기 때문에 영업일 2일 전은 토요일과 일요일이 포함되지 않은 목요일로 결과를 산출하게 된다. 

In [5]:
dt = pd.to_datetime( '2023-2-05' ) 
print ( '원래 값: ' , dt, dt.day_name()) 
print ( 'DateOffset 사용: ' , dt + BusinessDay(-2), (dt + BDay(-2)).day_name())

원래 값:  2023-02-05 00:00:00 Sunday
DateOffset 사용:  2023-02-02 00:00:00 Thursday


dateoffset을 사용하여 날짜와 시간의 연산은 다음과 같다. 

### dateoffset의 더하기, 빼기

시간의 더하기와 빼기는 더하거나 뺄 시간의 오프셋을 DateOffset으로 설정한 후 '+', '-' 연산 기호를 사용하여 계산한다. 

In [37]:
from pandas.tseries.offsets import * 
dt = pd.to_datetime( '2023-2-05' ) 
print ( '원래 값: ' , dt) 
print ( 'DateOffset 사용: ' , dt + DateOffset(weeks= 2 ))

원래 값:  2023-02-05 00:00:00
DateOffset 사용:  2023-02-19 00:00:00


앞의 코드를 보면 먼저 pandas.tseries.offsets 모듈에서 pandas와 DateOffset 클래스를 로딩 한 다음 to_datetime()를 사용핸 만든 datetime 개체에 하루의 DateOffset을 추가하고 결과 Timestamp 개체를 출력했다.
다음은 앞선 예와는 반대로 2개월 앞선 날을 산출하는 코드이다. 

In [3]:
dt = pd.to_datetime( '2023-2-05' ) 
print ( '원래 값: ' , dt) 
print ( 'DateOffset 사용: ' , dt + DateOffset(months= -2))

원래 값:  2023-02-05 00:00:00
DateOffset 사용:  2022-12-05 00:00:00


### dateoffset의 벡터 연산

DateOffset은 시점 데이터인 스칼라 시점 데이터 뿐만 아니라 DatetimeIndex와 같이 벡터 데이터에 대한 연산도 가능하다.  

In [21]:
pd.date_range(start = '2023-02-05', periods = 10, freq = 'D') + DateOffset(weeks = 10)

DatetimeIndex(['2023-04-16', '2023-04-17', '2023-04-18', '2023-04-19',
               '2023-04-20', '2023-04-21', '2023-04-22', '2023-04-23',
               '2023-04-24', '2023-04-25'],
              dtype='datetime64[ns]', freq=None)

### 특정 월의 시작일과 끝일의 산출

Timestamp로 설정된 특정 시점 월의 시작 일과 마지막 일을 산출하기 위해서는 DateOffset의 `MonthBegin()`과 `MonthEnd()`를 산술 연산자인 '+', '-'로 적용하여 사용할 수 있다. `MonthBegin()`에 '+' 연산이 적용되면 다음 월의 시작 일, '-' 연산이 적용되면 해당월의 시작 일이 리턴되고, `MonthEnd()`에 '+' 연산이 적용되면 해당 월의 마지막 일, '-' 연산을 적용하면 전 월의 마지막 일이 리턴된다.  

In [25]:
pd.Timestamp("2023/02/05") + pd.offsets.MonthBegin()

Timestamp('2023-03-01 00:00:00')

In [26]:
pd.Timestamp("2023/02/05") - pd.offsets.MonthEnd()

Timestamp('2023-01-31 00:00:00')

### 사용자 정의 영업일 설정

DateOffset의 서브 클래스인 BusinessDay에서 디폴트로 설정된 영업일은 월요일부터 금요일까지이다. 하지만 경우에 따라 영업일이 이와 다를 수 있다. 이를 설정하기 위해 사용하는 함수는 `CustomBusinessDay()`, 매개변수는 'weekmask'이다. 예를 들어 특정 업체의 휴일이 목요일과 금요일이라면 'weekmask'에 월, 화, 수, 토, 일요일을 리스트로 다음과 같이 설정한다. 

In [34]:
dt = pd.to_datetime('2023-2-06') 
print ( '원래 값: ' , dt, dt.day_name()) 
print ( 'DateOffset 사용: ', dt + pd.offsets.CustomBusinessDay(3, weekmask='Mon Tue Thu Sat Sun'), 
      (dt + pd.offsets.CustomBusinessDay(3, weekmask='Mon Tue Thu Sat Sun')).day_name())

원래 값:  2023-02-06 00:00:00 Monday
DateOffset 사용:  2023-02-11 00:00:00 Saturday


### 휴일의 설정

앞서 사용했던 BusinessDay의 하나 추가적인 기능은 휴일을 설정할 수 있다는 것이다. 휴일의 설정은 `CustomBusinessDay()`의 'holiday'를 사용한다. 다음은 휴일이 설정된 영업일을 설정하고 이에 대한 DataOffset를 사용한 예이다. 

In [35]:
dt = pd.to_datetime('2023-2-06') 
print ( '원래 값: ' , dt, dt.day_name()) 
print ( 'DateOffset 사용: ', dt + pd.offsets.CustomBusinessDay(3, weekmask='Mon Tue Thu Sat Sun', holidays = ['2023-02-11', '2023-02-12']), 
      (dt + pd.offsets.CustomBusinessDay(3, weekmask='Mon Tue Thu Sat Sun', holidays = ['2023-02-11', '2023-02-12'])).day_name())

원래 값:  2023-02-06 00:00:00 Monday
DateOffset 사용:  2023-02-13 00:00:00 Monday


## Timedelta을 사용한 연산

Timedelta는 Python의 datetime에서 제공하는 timedelta 클래스와 호환되는 pandas의 클래스로써, 두 Timestamp 개체 간의 시간적 차이를 나타낸다. 따라서 datetime.timedelta와 상호 호환 가능한 클래스이다. Timedelta는 pandas의 `Timedelta()`를 사용하여 생성 가능한데, 시간차의 기간과 기간 단위를 명기하거나 'day', 'month' 등의 키워드를 사용한 문자열(kwargs)로 생성이 가능하다. 또 두 개의 Timestamp간에 대해 '+', '-'의 산술 연산자를 사용하여 생성할 수도 있다. 하지만 Timedelta를 사용할 때 하나 주의해야 할 것은 앞서 사용했던 시간 주기 중 일부가 제공되지 않는다는 것이다. 가장 대표적인 것이 연과 월에 해당하는 'Y', 'M'이나 'years', 'months'가 제공되지 않는다. Timedelta에서 제공되는 시간 주기는 다음과 같다. 

- ‘W’, ‘D’, ‘T’, ‘S’, ‘L’, ‘U’, or ‘N’
- ‘days’ or ‘day’
- ‘hours’, ‘hour’, ‘hr’, or ‘h’
- ‘minutes’, ‘minute’, ‘min’, or ‘m’
- ‘seconds’, ‘second’, or ‘sec’
- ‘milliseconds’, ‘millisecond’, ‘millis’, or ‘milli’
- ‘microseconds’, ‘microsecond’, ‘micros’, or ‘micro’
- ‘nanoseconds’, ‘nanosecond’, ‘nanos’, ‘nano’, or ‘ns’

kwargs로 사용이 가능한 키워드는 days, seconds, microseconds, milliseconds, minutes, hours, weeks이다.  

In [12]:
print(pd.Timedelta(1, "W"))

7 days 00:00:00


In [9]:
print(pd.Timedelta(months = 10))

ValueError: cannot construct a Timedelta from the passed arguments, allowed keywords are [weeks, days, hours, minutes, seconds, milliseconds, microseconds, nanoseconds]

In [15]:
pd.Timestamp(2023,2,3) - pd.Timestamp(2023,1,3)

Timedelta('31 days 00:00:00')

### Timedelta의 더하기와 빼기

Timedelta를 사용하여 날짜와 시간의 더하기와 빼기는 더하거나 뺄 시간 간격을 Timedelta로 설정한 후 이를 '+', '-'로 계산한다. 

In [38]:
pd.Timestamp(2023,2,3) + pd.Timedelta(1, "W")

Timestamp('2023-02-10 00:00:00')

In [40]:
pd.Timestamp(2023,2,3) + pd.Timedelta(-1, "W")

Timestamp('2023-01-27 00:00:00')

### Timedelta 간의 간격 계산

특정한 두 개의 날짜나 시간 간의 간격을 계산하기 위해 Timedelta가 사용된다. Timestamp로 설정된 두 개의 시점을 '-' 연산으로 계산하면 그 두 시점간의 간격이 Timedelta 클래스로 저장된다.   

In [52]:
pd.Timestamp('2023-2-5') - pd.Timestamp('2023-1-5')

Timedelta('31 days 00:00:00')

 이 연산은 단순한 시점간의 스칼라 연산뿐 아니라 Series나 배열과 같은 벡터 연산도 가능하다. 

In [48]:
dates1 = pd.date_range("2023-02-03", periods = 5, freq = 'D')
dates2 = pd.date_range("2023-02-03", periods = 5, freq = '3D')

df = pd.DataFrame(data={
    "date_col1":dates1, "date_col2":dates2})

df

,date_col1,date_col2
0,2023-02-03,2023-02-03
1,2023-02-04,2023-02-06
2,2023-02-05,2023-02-09
3,2023-02-06,2023-02-12
4,2023-02-07,2023-02-15


In [49]:
df["date_col2"] - df["date_col1"]

0   0 days
1   2 days
2   4 days
3   6 days
4   8 days
dtype: timedelta64[ns]

### Timedelta의 곱하기와 나누기

Timedelta 객체는 곱하기 연산과 나누기 연산이 가능하다. 곱하기 연산과 나누기 연산은 산술 연산자인 '\*'와 '\\' 연산자에 정수를 적용함으로써 사용이 가능하다. Timedelta 객체에 정수를 곱하거나 나누는 것은 주로 Timedelta 객체의 간격을 조정하는 데 사용된다. 예를 들어, Timedelta 객체의 간격을 몇 배 늘리거나 줄이거나, 일주일을 반으로 나누어 새로운 기간을 계산하는 등의 작업에 유용하게 사용된다.

In [93]:
print(pd.Timedelta(days=2) * 3)

6 days 00:00:00


In [8]:
print(pd.Timedelta(days=7) / 3)

2 days 08:00:00


이 연산을 잘 활용하면 특정 시간 주기를 다른 주기로 계산하는데도 활용이 가능하다. 예를 들어 특정한 두 개의 날짜나 시간의 간격을 시, 분, 초 단위의 주기로 계산할 때 유용하게 사용된다. 이는 특정 시간 주기를 산출하고자 하는 Timestamp 단위로 나누어주면 산출할 수 있다. 다음은 2월 7일 13시 44분에서 2월 3일 11시 12분까지의 간격을 분으로 계산하는 코드이다.   

In [15]:
(pd.Timestamp('2023-02-07 13:44') - pd.Timestamp('2023-02-03 11:12')) / pd.Timedelta(minutes = 1)

5912.0

### Timedelta의 최댓값과 최솟값

Timedelta 클래스에서 시간의 간격을 저장하는 데에는 최댓값과 최솟값이 있다. 최댓값은 `Timedelta.max`로 산출할 수 있는데, 106,751일, 23:47:16.854775807로 약 292년 정도이며, 파이썬의 datetime 모듈에서 지원하는 최대 시간 간격인 999999999 days, 23:59:59.999999보다는 작다. 또 최솟값은 `Timedelta.min`으로 산출할 수 있는데, -10,6752일, 00:12:43.145224193로, 파이썬의 datetime 모듈에서 지원하는 최소 시간 간격인 -999999999 days, 0:00:00보다 작다. 사실 datetime 모듈에서 지원하는 최대 간격은 275만년으로 오스트랄로 피테쿠스가 활동하던 때라 큰 의미가 없을 듯 하다.

In [59]:
print('최솟값 : ', pd.Timedelta.min, ', ' '최댓값 : ', pd.Timedelta.max)

최솟값 :  -106752 days +00:12:43.145224193 , 최댓값 :  106751 days 23:47:16.854775807


# 날짜와 시간의 슬라이싱

python에서 다량의 데이터를 사용하기 위해서는 데이터를 행열 형태로 구성된 테이블로 가져와서 분석해야 한다. 여기에 가장 흔히 사용되는 데이터 클래스가 pandas에서 지원하는 데이터프레임이다. 사실 pandas의 데이터프레임은 pandas에서 지원하는 여러개의 Series 데이터 클래스가 열로 구성된 데이터 객체이다. 여러 Series가 묶여야하기 때문에 이들은 index라는 공통된 데이터 속성에 의해 참조된다. 이 index가 날짜나 시간으로 구성되면 시계열 데이터가 되는 것이다. 

이러한 panas 데이터프레임에서 제공하는 다양한 기능 중에 매우 많이 사용되는 기능이 슬라이스이다. 데이터프레임 중 일부만을 사용하기 위해 index의 일부에 해당하는 데이터만을 선택하는 것을 슬라이싱이라고 한다. 데이터프레임 슬라이싱의 결과로 산출되는 것은  numpy arrays나 Python lists로 생성된다.   

## 특정 시점 데이터 슬라이싱

시계열 데이터프레임에서 특정 시점의 데이터를 슬라이싱 하기 위해서는 `loc[]`을 사용한다. `loc[]`에 슬라이싱하고자하는 특정 날짜나 시간을 명기하면 해당 날짜의 데이터에 해당하는 행 전체를 슬라이싱 할 수 있다. 만약 이중 특정 열만을 슬라이싱하기 위해서는 콤마(,)를 사용하여 열 리스트를 설정한다.  

In [18]:
df_GDP.loc['2021', ['Canada', 'Korea, Rep.']]

country,Canada,"Korea, Rep."
year,,
2021-01-01,43936.308105,32730.650628


In [27]:
df_import.loc['2022-10-01', df_import.columns.get_level_values(1)=='무역수지']

국가    항목  
중국    무역수지   -1254397
미국    무역수지    1908107
일본    무역수지   -2028126
홍콩    무역수지    1814784
베트남   무역수지    2848768
대만    무역수지    -445621
싱가포르  무역수지     529239
인도    무역수지     881504
독일    무역수지   -1389699
멕시코   무역수지     144437
Name: 2022-10-01 00:00:00, dtype: int64

In [29]:
df_gold.loc['2022-09-02']

Open         1707.9
High        1729.45
Low         1707.05
Close        1723.0
Volume            0
Currency        USD
Name: 2022-09-02 00:00:00, dtype: object

## 시점 전체 슬라이싱

시계열 데이터를 슬라이싱 할 때는 앞서 설명한 특정 시점 데이터를 슬라이싱 할수도 있겠지만 연 전체, 월 전체와 같이 특정 날짜와 시간 주기 전체를 슬라이싱 할 경우가 더 많다. 이런 경우는 슬라이싱 하고자 하는 주기의 시점까지의 문자열로 슬라이싱하면 해당 주기에 해당하는 데이터가 모두 슬라이싱된다. 예를 들어 월간 데이터인 df_import에서 2022년 데이터를 모두 슬라이싱하려면 `loc('2022')`를 사용한다.   

In [34]:
df_import.loc['2022', df_import.columns.get_level_values(0)=='중국'].tail(3)

국가                중국                   
항목              수출금액      수입금액     무역수지
2022-10-01  12157489  13411886 -1254397
2022-11-01  11376108  12129628  -753520
2022-12-01  11185579  11805167  -619588

In [38]:
df_gold.loc['2022-08'].head(3)

,Open,High,Low,Close,Volume,Currency
Date,,,,,,
2022-08-01,1782.5,1792.5,1774.3,1787.7,133742,USD
2022-08-02,1789.2,1805.0,1776.0,1789.7,174165,USD
2022-08-03,1777.3,1789.4,1770.0,1776.4,151734,USD


## 시점 범위 슬라이싱

시계열 데이터의 슬라이싱에는 특정 날짜와 시간의 범위에 해당하는 슬라이싱이 필요할 때가 많다. pandas의 데이터프레임에서 이러한 범위를 설정하여 슬라이싱할 때는 ':'을 사용한다. 날짜와 시간에서도 ':'을 사용하여 시작 시점과 끝 시점의 범위를 설정하고 슬라이싱할 수 있다. 다음은 날짜와 시간의 범위를 슬라이싱하는 예이다. 

In [41]:
df_GDP.loc['2015':'2021', ['Canada', 'Korea, Rep.']]

country,Canada,"Korea, Rep."
year,,
2015-01-01,43596.135537,28732.231076
2016-01-01,43536.913403,29461.784007
2017-01-01,44325.488337,30307.395236
2018-01-01,44917.483728,31053.637929
2019-01-01,45109.244486,31640.214630
2020-01-01,42258.691017,31372.471261
2021-01-01,43936.308105,32730.650628


In [50]:
df_import.loc['2022-10':'2022-12', df_import.columns.get_level_values(1)=='무역수지']

국가,중국,미국,일본,홍콩,베트남,대만,싱가포르,인도,독일,멕시코
항목,무역수지,무역수지,무역수지,무역수지,무역수지,무역수지,무역수지,무역수지,무역수지,무역수지
2022-10-01,-1254397,1908107,-2028126,1814784,2848768,-445621,529239,881504,-1389699,144437
2022-11-01,-753520,2251219,-2002736,1690837,2699269,-46894,549070,842734,-1358157,196901
2022-12-01,-619588,2584020,-2045414,1649436,2891755,-365168,513669,992346,-1220566,43042


In [49]:
df_gold.loc['2022-08-02':'2022-08-06']

,Open,High,Low,Close,Volume,Currency
Date,,,,,,
2022-08-02,1789.2,1805.0,1776.0,1789.7,174165,USD
2022-08-03,1777.3,1789.4,1770.0,1776.4,151734,USD
2022-08-04,1782.1,1812.0,1779.4,1806.9,153292,USD
2022-08-05,1807.7,1811.6,1780.2,1791.2,168901,USD


# 날짜와 시간의 시프트

시계열 데이터가 다른 데이터와 가장 큰 차이는 날짜와 시간의 인덱스가 반드시 필요하다는 것이다. 이 날짜와 시간은 다른 데이터와는 달리 데이터의 순서가 정해져 있다는 특징이 존재한다. 다른 데이터 들도 데이터가 발생한 순서가 중요할 수 있지만 일반적으로는 그 순서 자체가 큰 의미가 없다. 하지만 시계열 데이터는 시간에 따라 순서가 정해져 있기 때문에 이 순서에 따른 연산도 가능하다. 

시계열 데이터의 순서에 따른 대표적인 연산이 lag 연산이다. lag는 시계열 데이터에서 해당 데이터의 시점과 이전 시점 데이터를 비교하기 위해 사용하는 지연을 말한다.  lag는 시계열 데이터가 가지고 있는 일정한 주기의 과거 시점 데이터와의 패턴을 파악하고 예측하기 위해 중요한 연산이다. 특히 전년대비 증감율과 같은 데이터를 산출하기 위해서는 lag가 반드시 필요하다. 

python에서 lag 연산을 위해서는 pandas의 `shift()`를 사용한다. `shift()` 함수는 시계열 데이터의 각 원소를 주어진 기간만큼 이동시키는 기능을 제공하는데, 매개변수로 'periods'와 'freq'를 설정할 때 결과가 달라진다. 다음은 `shift()`에서 사용하는 주요 매개변수이다. 
<div class="alert alert-success">
    
- periods : 이동시킬 기간을 정수로 지정, 양수인 경우 이전 방향으로 이동하며, 음수인 경우 다음 방향으로 이동
- freq : 시계열 데이터의 주기를 지정, 이 매개변수를 사용하면 시간 주기로 shift가 수행됨
- axis : shift 연산이 수행될 축의 지정, 0은 행 방향으로, 1은 열 방향으로 shift 됨
- fill_value : 누락 데이터의 대체 값 지정
    
</div>

## period를 사용한 시프트

시계열 데이터는 그 데이터의 특성에 따라 일부 구간의 누락이 발생할 수 있다. 다만 이 누락이 데이터 자체적으로 인정되는 누락인지, 데이터가 손실되어 발생한 누락인지의 여부가 중요하다. 예를 들어 특정 점포의 판매량 데이터의 경우 해당 점포가 영업하지 않는 휴일의 경우에는 판매량 데이터가 누락되게 된다. 그러나 영업을 했음에도 판매량 데이터가 없다면 이는 데이터의 손실에 따른 비정상적인 데이터 손실로 봐야 할 것이다. 

이렇게 발생된 데이터들의 지연(lag) 연산을 하기 위해서는 `shift()`의 'periods' 매개변수를 사용해야 한다. 'periods'는 날짜와 시간의 간격에 상관없이 발생된 데이터들의 lag를 발생시킨다. 

다음은 1회차, 2회차의 lag가 발생된 두 개의 데이터프레임을 보이고 있다. 첫 번째 데이터에서는 연 간격으로 구성된 데이터이고 누락된 값이 없기 때문에 lag된 데이터도 누락없이 생성되었다. 그러나 두 번째 데이터는 주말에 해당하는 1월 8일과 1월 9일 데이터가 누락되었지만, 이와는 무관하게 데이터들이 lag 되었다.    

In [64]:
pd.DataFrame({'origin' : df_GDP['Korea, Rep.'], 'shifted_1' : df_GDP['Korea, Rep.'].shift(periods = 1), 
              'shifted_2' : df_GDP['Korea, Rep.'].shift(periods = 2)}).head(5)

,origin,shifted_1,shifted_2
year,,,
1960-01-01,1027.468538,NaN,NaN
1961-01-01,1066.610515,1027.468538,NaN
1962-01-01,1076.920824,1066.610515,1027.468538
1963-01-01,1141.820659,1076.920824,1066.610515
1964-01-01,1217.726265,1141.820659,1076.920824


In [66]:
pd.DataFrame({'origin' : df_gold['Close'], 'shifted_1' : df_gold['Close'].shift(periods = 1, fill_value=0), 
              'shifted_2' : df_gold['Close'].shift(periods = 2, fill_value=0)}).head(5)

,origin,shifted_1,shifted_2
Date,,,
2000-01-04,283.7,0.0,0.0
2000-01-05,282.1,283.7,0.0
2000-01-06,282.4,282.1,283.7
2000-01-07,282.9,282.4,282.1
2000-01-10,282.7,282.9,282.4


## freq를 사용하는 시프트

앞선 금 가격의 데이터에서 2000년 1월 8, 9일 데이터는 누락되어 있다. 이 두 날은 토요일과 일요일에 해당하는 날로써 금 거래소에서 거래가 이루어지지 않은 날이다. 만약 이 날짜를 포함하여 lag 연산을 하기 위해서는 'periods' 매개변수 대신 'freq' 매개변수를 사용해야 한다. 'freq' 매개변수에는 을 설정할 수 있는데 'freq'에는 DateOffset, timedelta, 시간의 간격 문자열을 설정할 수 있다. 

다음은 일 간격으로 lag를 산출하는 `shift()`의 활용 예이다. 다음의 코드에서는 1일 간격으로 lag를 산출하는데 이로 인해 원 데이터에 없던 날짜인 2000년 1월 8일이 나타났다. 이 날은 1월 7일 데이터가 이동한 날이기 때문에 원 데이터 열에는 데이터가 없을을 뜻하는 'NaN'이 표현되었고 lag된 데이터는 1월 7일이었던 데이터가 표현되었다. 또 2000년 1월 10일의 데이터의 경우 일 기준의 lag 값은 1월 9일 데이터가 없기 때문에 'NaN'으로 표시되었다.  

In [76]:
pd.DataFrame({'origin' : df_gold['Close'], 'shifted_D' : df_gold['Close'].shift(freq='D')}).head(10)

,origin,shifted_D
Date,,
2000-01-04,283.7,NaN
2000-01-05,282.1,283.7
2000-01-06,282.4,282.1
2000-01-07,282.9,282.4
2000-01-08,NaN,282.9
2000-01-10,282.7,NaN
2000-01-11,284.4,282.7
2000-01-12,283.7,284.4
2000-01-13,285.1,283.7


'fraq'를 사용하는 방법 중 특이한 방법은 'infer' 방법이다. ..............................................

# 날짜와 시간의 샘플링과 보간

앞에서 설명했듯이 시계열 데이터 분석은 날짜 혹은 시간에 따라 기록된 분석 대상 데이터의 흐름과 추세를 패턴화해서 분석하는 과정이다. 이러한 분석을 위해서는 단순히 날짜와 시간이 포함된 시계열 데이터가 아닌 분석이 가능하도록 일정한 간격으로 기록된 데이터가 필요하다. 예를 들어 특정 상품의 판매 데이터라고 한다면 상품이 판매될 당시의 날짜와 시간이 기록된 시계열 데이터가 만들어지겠지만 이 데이터를 통계적 시계열 분석 알고리즘이나 머신러닝 시계열 분석 알고리즘을 적용하기 위해서는 일정한 간격과 주기로 데이터가 기록되어야 한다. python에서는 이와 같이 일정한 간격으로 데이터를 기록하는 것을 샘플링이라고 한다. 

이 샘플링은 업샘플링(Upsampling)과 다운샘플링(Downsampling)의 두 가지가 있다. 

업샘플링은 데이터가 수집된 주기보다 더 작은 주기로 데이터를 만드는 것을 의미한다. 연도별로 저장된 데이터를 월별 데이터나 일별 데이터로 변환하는 것이 업샘플링이다. 업샘플링의 경우는 실제 데이터보다 더 많은 수의 데이터가 필요하기 때문에 새로 발생한 데이터의 값을 채우는 방법이 필요하다. 이 방법을 보간(Interpolate)이라고 한다. 

다운샘플링은 데이터가 수집된 주기보다 더 큰 주기로 데이터를 만드는 것을 의미한다. 일별로 저장된 데이터를 월별 데이터나 년도별 데이터로 변환하는 것이 다운샘플링이다. 다운샘플링은 많은 간격의 데이터를 적은 간격의 데이터로 만드는 것이기 때문에 데이터를 요약하는 방법이 필요하다. 

## 샘플링의 재설정

날짜와 시간이 DatetimeIndex로 기록된 데이터프레임의 경우 기록된 날짜와 시간의 간격이 같이 기록된다. 이 간격은 DatatimeIndex가 생성될때 지정되어야 하고, 지정되지 않으면 간격이 설정되지 않는다. 앞서 불러들여온 세 개의 시계열 데이터의 시간 인덱스 주기는 다음과 같이 설정되지 않았다. 

In [82]:
print(df_GDP.index.freq)

None


In [83]:
print(df_import.index.freq)

None


In [84]:
print(df_gold.index.freq)

None


이렇게 설정되지 않은 시계열 데이터에 간격을 설정하기 위해서 `asfreq()`를 사용한다. 

In [101]:
df_GDP.asfreq('Y', method = 'ffill').info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 61 entries, 1960-12-31 to 2020-12-31
Freq: A-DEC
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Canada          24 non-null     float64
 1   European Union  51 non-null     float64
 2   France          61 non-null     float64
 3   Germany         51 non-null     float64
 4   Italy           61 non-null     float64
 5   Japan           61 non-null     float64
 6   Korea, Rep.     61 non-null     float64
 7   United Kingdom  61 non-null     float64
 8   United States   61 non-null     float64
 9   World           61 non-null     float64
dtypes: float64(10)
memory usage: 5.2 KB


In [100]:
df_GDP.asfreq('Y', method = 'ffill').tail(5)

country,Canada,European Union,France,Germany,Italy,Japan,"Korea, Rep.",United Kingdom,United States,World
year,,,,,,,,,,
2016-12-31,43536.913403,31021.279373,36956.795800,41682.032243,30685.645899,35242.199377,29461.784007,45699.296963,57292.538783,10317.038662
2017-12-31,44325.488337,31851.783773,37694.083303,42639.554409,31244.227099,35861.972855,30307.395236,46499.013546,58207.578310,10544.907045
2018-12-31,44917.483728,32452.667830,38259.697770,42928.741237,31593.480262,36117.227690,31053.637929,47006.142927,59607.393660,10772.498276
2019-12-31,45109.244486,33011.437074,38832.024201,43284.602455,32114.299856,36081.065312,31640.214630,47491.594563,60698.011299,10936.163035
2020-12-31,42258.691017,31114.573597,35712.075199,41650.422713,29353.815443,34556.439929,31372.471261,42098.749784,58453.446128,10489.381172


## 업샘플링(upsampling)

- ‘Forward filling’ or ‘padding’ — Use the last known value to fill the new one.
- ‘Backfilling’ — Use the next known value to fill the new one.
You can also fill with NaNs using the asfreq() function with no arguments. This will result in new data points having NaNs in them, which you can use a fillna() function on later.

## 다운샘플링(downsampling)

## 날짜와 시간의 윈도우 연산

주식 투자를 해본 사람이라면 주식 가격의 변화에 대한 시계열 그래프를 자주 보게 된다. 이 그래프에 특이한 점이 있다면 보통의 경우 이동 평균이 같이 표시된다는 것이다. 이동 평균이란 특정 시점을 기점으로 일정 과거 기간동안의 평균을 의미하는 것으로 특정 시점이 점차 과거로 갈수록 이동 평균의 기간도 일정하게 과거로 이동하게 된다. 이러한 이동 평균은 계열 데이터의 이동성과 추세를 판단하는데 중요한 통계 자료가 된다. 

이동 평균과 같이 일정한 기간을 대상으로 한 연산을 윈도우 연산이라고 한다. 윈도우 연산은 전체 시계열 데이터를 다시 윈도우 크기로 그룹화하고, 이 그룹의 데이터로 산출한 요약 통계치로 구성된 새로운 시계열 데이터를 생성한다. 이렇게 윈도우 연산이 적용된 시계열 데이터를 원 시계열 데이터와 비교하여 시계열 데이터의 패턴과 추세를 파악한다. 윈도우 연산에 적용되는 요약 통계량은 주로 이동 평균, 이동 합계, 이동 표준편차 등을 사용하는데 최댓값, 최솟값, 분위수등을 사용할 수도 있다. 

<div class="alert alert-success">

- 이동 평균(Moving Average): 윈도우로 설정한 기간에 대한 데이터의 산술 평균으로, 시계열 데이터의 추세를 파악하는데 도움이 된다.
- 이동 합계(Moving Sum): 윈도우로 설정한 기간에 대한 데이터의 합계, 시계열 데이터의 추세를 파악하는데 도움이 된다.
- 이동 표준편차(Moving Standard Deviation): 윈도우로 설정한 기간에 대한 표준편차로, 시계열 데이터의 변동성(Volatility)을 파악하는 데 도움이 된다.

    </div>

윈도우 연산에서 가장 중요한 것은 일정한 크기의 윈도우(Window)를 설정하는 것이다. 윈도우 크기는 분석하려는 데이터의 특성과 목적에 따라 설정할 수 있으며, 일반적으로는 고정된 크기의 윈도우를 사용하지만, 가변적인 크기의 윈도우도 사용된다.


### rollding windows

In [ ]:
df_

### expanding windows

# 날짜와 시간의 그룹화 및 요약